# appendix E  Batching and throughput-oriented execution

# E.2 Running batched generation

In [1]:
import torch

from reasoning_from_scratch.ch02 import (
    get_device,
    generate_text_basic_stream_cache,
)
from reasoning_from_scratch.ch03 import (
    load_model_and_tokenizer,
    render_prompt,
)

device = get_device()
model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

for problem in ["2+2?", "3+3=6?"]:
    prompt = render_prompt(problem)
    input_ids = torch.tensor(
        tokenizer.encode(prompt),
        dtype=torch.long,
        device=device,
    ).unsqueeze(0)

    for token in generate_text_basic_stream_cache(
        model=model,
        token_ids=input_ids,
        max_new_tokens=32,
        eos_token_id=tokenizer.eos_token_id,
    ):
        next_token_id = token.squeeze(0)
        print(tokenizer.decode(next_token_id.tolist()), end="", flush=True)

    print()

Using CPU
✓ qwen3\qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


# Listing E.2 Batched generatio

In [5]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3\qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


In [4]:
from reasoning_from_scratch.qwen3_batched import (
    generate_text_basic_batched_cache,
    load_model_and_tokenizer,
)

model, tokenizer = load_model_and_tokenizer(
    which_model="base",
    device=device,
    use_compile=False,
)

problems = ["2+2?", "3+3=6?"]
prompts = [render_prompt(problem) for problem in problems]
tokenized = [tokenizer.encode(p) for p in prompts]
pad_id = tokenizer.pad_token_id
max_len = max(len(t) for t in tokenized)

left_padded = [
    [pad_id] * (max_len - len(t)) + t
    for t in tokenized
]
input_ids = torch.tensor(left_padded, dtype=torch.long, device=device)

generated = generate_text_basic_batched_cache(
    model=model,
    token_ids=input_ids,
    max_new_tokens=32,
    eos_token_id=tokenizer.eos_token_id,
    pad_id=pad_id,
)

for row in generated:
    eos_pos = (row == tokenizer.eos_token_id).nonzero(as_tuple=True)[0]
    if len(eos_pos) > 0:
        row = row[:eos_pos[0]]
    print(tokenizer.decode(row.tolist()))

✓ qwen3\qwen3-0.6B-base.pth already up-to-date
 \boxed{4}
 \boxed{6}


# E.3 Padding and attention masks

In [6]:
input_ids = torch.tensor([[17, 10, 17, 30]])

# E.4 Chapter 3: Batched MATH-500 evaluation

In [ ]:
from reasoning_from_scratch.ch07 import download_from_github

download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500.py"
)
download_from_github(
    "ch03/01_main-chapter-code/math500_test.json",
    out="math500_test.json",
)
# uv run evaluate_math500.py \
#   --dataset_size 500 \
#   --which_model "reasoning"

evaluate_math500.py: 3.5 KB (cached)
math500_test.json: 466.1 KB (cached)


WindowsPath('math500_test.json')

In [ ]:
download_from_github(
    "ch03/02_math500-verifier-scripts/evaluate_math500_batched.py"
)

# uv run evaluate_math500_batched.py \
#   --dataset_size 500 \
#   --which_model "reasoning" \
#   --batch_size 64

evaluate_math500_batched.py: 8.3 KB


# E.5 Chapter 4: Batched self-consistency sampling

In [9]:
download_from_github(
    "ch04/02_math500-inference-scaling-scripts/"
    "self_consistency_math500_batched.py"
)

# uv run self_consistency_math500_batched.py \
#   --which_model base \
#   --temperature 0.9 \
#   --top_p 0.9 \
#   --num_samples 3 \
#   --dataset_size 500 \
#   --prompt_suffix "\n\nExplain step by step."

self_consistency_math500_batched.py: 12.3 KB


# E.6 Chapter 6: Batched GRPO rollouts

In [10]:
download_from_github(
    "ch06/02_rlvr_grpo_scripts_intro/"
    "rlvr_grpo_original_no_kl_batched.py"
)

# uv run rlvr_grpo_original_no_kl_batched.py \
#   --num_rollouts 8 \
#   --batch_size 4 \
#   --max_new_tokens 512

rlvr_grpo_original_no_kl_batched.py: 14.7 KB


# E.7 Chapter 8: Batched distillation

In [11]:
from reasoning_from_scratch.ch08 import load_distill_data

download_from_github(
    "ch08/04_train_with_distillation/distill_batched.py"
)
_ = load_distill_data(
    partition="deepseek-r1-math-train",
    local_path="deepseek-r1-math-train.json",
)

# uv run distill_batched.py \
#   --data_path deepseek-r1-math-train.json \
#   --dataset_size 500 \
#   --validation_size 10 \
#   --epochs 2 \
#   --use_think_tokens \
#   --batch_size 32

distill_batched.py: 18.5 KB
deepseek-r1-math-train.json: 107610.0 KB (cached)
